# 08 · Export for the dashboard, and the model card

**Out:** `dashboard/public/model/`, the only data the dashboard reads. Every number on
the page comes from these files, which come from the notebooks; nothing is typed by hand.

| file | what it holds |
|---|---|
| `model.json` | the final LightGBM trees, category codes and feature order |
| `cells.json` | each 0.25° cell's 365-day accident count and smoothed rate, as of March 2023 |
| `holidays.json` | US federal holidays, the calendar the features use |
| `calibration.json` | probability calibration and conformal thresholds |
| `fixtures.json` | test accidents with Python's scores and SHAP values, for the parity tests |
| `evidence.json` | model comparison, test metrics, backtest, slices, decisions |
| `story.json` | the forensic findings from notebooks 02 and 03 |
| `explanations.json` | global explanations from notebook 07 |
| `scenarios.json` | real test accidents chosen by rule, as starting points for the explorer |
| `model_card.json` | intended use, data, metrics, limitations, licence |

In [1]:
import json

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMClassifier

from accidents import benchmark, calibrate, clean, config, export, features, ingest, split

FINAL = config.ROOT / "data" / "processed" / "final"
STORY = config.ROOT / "data" / "processed" / "story"
PUBLIC = config.ROOT / "dashboard" / "public" / "model"
PUBLIC.mkdir(parents=True, exist_ok=True)


def read(path):
    return json.loads(path.read_text())


def write(name, payload):
    (PUBLIC / name).write_text(json.dumps(payload, separators=(",", ":")))


frozen = read(FINAL / "frozen.json")
calibrator = read(FINAL / "calibrator.json")
conformal = read(FINAL / "conformal.json")
metrics = read(FINAL / "test_metrics.json")
explanations = read(FINAL / "explanations.json")
selection = read(benchmark.OUT / "selection.json")
leaderboard = read(benchmark.OUT / "leaderboard.json")
model = joblib.load(FINAL / "model.joblib")["model"]
assert isinstance(model, LGBMClassifier), "the dashboard scorer implements LightGBM trees"
print("exporting", frozen["model"], "|", frozen["final_fit"], "|", frozen["calibration"])

exporting lightgbm | refit on all training months | platt


## 1 · The model and its lookups

In [2]:
write("model.json", export.lightgbm_trees(model))

population = clean.mask_impossible_readings(
    clean.target_window(ingest.read_parquet(ingest.POPULATION_COLUMNS, ingest.POPULATION_ROWS))
)
train_rows = split.assign_split(population["Start_Time"]) == "train"
prior_rate = float(population.loc[train_rows, "y"].mean())
cells = export.cell_history(population, prior_rate, as_of=config.DATA_END)
write("cells.json", cells)
write("holidays.json", export.holidays())
write(
    "calibration.json",
    {"calibrator": calibrator, "conformal": conformal, "cost_ratios": [5, 10, 20, 50]},
)
print(f"{len(cells['cells']):,} grid cells with history; prior rate {prior_rate:.4f}")

6,826 grid cells with history; prior rate 0.0246


## 2 · Parity fixtures

300 random test accidents with Python's log-odds, calibrated probability and SHAP
values. `dashboard/tests/unit/trees.test.ts` requires the TypeScript scorer to match them
to 1e-6.

In [3]:
table = pd.read_parquet(config.MODEL_TABLE)
test = table[table["split"] == "test"].reset_index(drop=True)
fixture_rows = test.sample(300, random_state=config.SEED)[features.FEATURES]
write("fixtures.json", {"rows": export.parity_fixtures(model, fixture_rows, calibrator)})

## 3 · Evidence and story

In [4]:
write("evidence.json", {
    "selection": selection,
    "frozen": frozen,
    "validation": leaderboard,
    "test": metrics,
})  # fmt: skip
write("story.json", {"eda": read(STORY / "eda.json"), "leakage": read(STORY / "leakage.json")})
write("explanations.json", explanations)

## 4 · Starting scenarios from real accidents

Six real test-month accidents, chosen by rule so that the explorer opens on genuine
conditions rather than invented ones: the highest-risk and lowest-risk accident, a snow or
ice accident, a night-time interstate accident, a weekday rush-hour accident, and the
median-risk accident. Each keeps its true outcome for comparison, and its area history as
of its own date, so the explorer shows the same probability as Python for it.

In [5]:
p_test = calibrate.apply(calibrator, model.predict_proba(test[features.FEATURES])[:, 1])
picks = {
    "highest predicted risk": int(np.argmax(p_test)),
    "lowest predicted risk": int(np.argmin(p_test)),
    "median predicted risk": int(np.argsort(p_test)[len(p_test) // 2]),
}
rules = {
    "snow or ice": test["weather"] == "snow_ice",
    "night on an interstate": (test["road_class"] == "interstate")
    & test["hour"].isin([0, 1, 2, 3, 4]),
    "weekday rush hour": (test["is_weekend"] == 0) & test["hour"].isin([7, 8, 16, 17]),
}
for label, mask in rules.items():
    candidates = np.flatnonzero(mask.to_numpy())
    picks[label] = int(candidates[np.argsort(p_test[candidates])[len(candidates) // 2]])
scenarios = []
for label, i in picks.items():
    row = test.iloc[i]
    scenarios.append({
        "label": label,
        "start_time": row["start_time"].strftime("%Y-%m-%dT%H:%M"),
        "lat": round(float(row["lat"]), 5),
        "lng": round(float(row["lng"]), 5),
        "state": str(row["state"]),
        "road_class": str(row["road_class"]),
        "weather": str(row["weather"]),
        "wind_direction": str(row["wind_direction"]),
        **{name: export.json_value(row[name]) for name in features.WEATHER_COLUMNS.values()},
        **{name: int(row[name]) for name in features.POI_COLUMNS.values()},
        # the area history as of this accident's date, so the explorer reproduces Python
        "cell_count": int(row["cell_count"]),
        "cell_rate": float(row["cell_rate"]),
        "probability": round(float(p_test[i]), 5),
        "actual": "major" if row["y"] == 1 else "minor",
    })  # fmt: skip
write("scenarios.json", scenarios)
display(
    pd.DataFrame(scenarios)[
        ["label", "start_time", "state", "road_class", "weather", "probability", "actual"]
    ]
)

,label,start_time,state,road_class,weather,probability,actual
0,highest predicted risk,2022-11-01T11:04,WI,state_route,clear,0.70679,minor
1,lowest predicted risk,2022-10-13T14:48,FL,local,rain,0.00031,minor
2,median predicted risk,2022-12-22T11:46,NC,interstate,cloudy,0.01101,minor
3,snow or ice,2022-12-23T08:30,NY,interstate,snow_ice,0.01589,minor
4,night on an interstate,2022-12-24T02:33,FL,interstate,cloudy,0.02563,minor
5,weekday rush hour,2023-02-22T07:11,TX,interstate,rain,0.01003,minor


## 5 · Model card

Following the sections of Mitchell et al., *Model Cards for Model Reporting* (2019).

In [6]:
test_scores = metrics["test"]
card = {
    "model": f"{frozen['model']} ({frozen['final_fit']}), {frozen['calibration']} calibration",
    "task": "Probability that a US traffic accident has a major impact on traffic "
    "(US-Accidents Severity 4 vs 2), from conditions known when it is first reported.",
    "intended_use": "Demonstrating and explaining a carefully evaluated risk model; exploring "
    "how place, time, weather and road context relate to major traffic impact.",
    "not_intended_for": "Dispatch, routing or any operational decision; inferring injury or "
    "fatality (the label measures traffic delay); comparing the safety of states, since part of "
    "the place signal reflects how agencies report closures.",
    "training_data": "Source1 records of US-Accidents (Moosavi et al., 2019), January 2021 – "
    "March 2022; October–December 2020 only feeds the location history.",
    "evaluation_data": "Validation April–September 2022 (model choice, calibration); test "
    "October 2022 – March 2023, scored once.",
    "features": features.FEATURES,
    "test_metrics": {k: test_scores[k] for k in ("average_precision", "ap_low", "ap_high",
                                                 "roc_auc", "log_loss", "brier", "ece")},
    "test_base_rate": metrics["test_base_rate"],
    "random_split_average_precision": metrics["random_split_ap"],
    "temporal_validation_average_precision": metrics["temporal_valid_ap"],
    "conformal_coverage_test": metrics["conformal_test_coverage"],
    "limitations": [
        "The label is the feed's own traffic-impact rating, not a measured delay.",
        "Place features carry reporting practice as well as road risk.",
        "Station pressure partly encodes altitude.",
        "Season effects rest on one to two years of training data.",
        "Probabilities are calibrated on April–September 2022; drift can erode calibration.",
    ],
    "data_licence": "US-Accidents is licensed CC BY-NC-SA 4.0. Cite: Moosavi et al., 'A "
    "Countrywide Traffic Accident Dataset' (2019) and 'Accident Risk Prediction based on "
    "Heterogeneous Sparse Data: New Dataset and Insights' (ACM SIGSPATIAL 2019). Derived "
    "files are shared under the same licence, for non-commercial use.",
}  # fmt: skip
write("model_card.json", card)
print(json.dumps(card, indent=2)[:2500])

{
  "model": "lightgbm (refit on all training months), platt calibration",
  "task": "Probability that a US traffic accident has a major impact on traffic (US-Accidents Severity 4 vs 2), from conditions known when it is first reported.",
  "intended_use": "Demonstrating and explaining a carefully evaluated risk model; exploring how place, time, weather and road context relate to major traffic impact.",
  "not_intended_for": "Dispatch, routing or any operational decision; inferring injury or fatality (the label measures traffic delay); comparing the safety of states, since part of the place signal reflects how agencies report closures.",
  "training_data": "Source1 records of US-Accidents (Moosavi et al., 2019), January 2021 \u2013 March 2022; October\u2013December 2020 only feeds the location history.",
  "evaluation_data": "Validation April\u2013September 2022 (model choice, calibration); test October 2022 \u2013 March 2023, scored once.",
  "features": [
    "lat",
    "lng",
    "te

## 6 · What ships

In [7]:
inventory = pd.DataFrame(
    [
        {"file": f.name, "kilobytes": round(f.stat().st_size / 1024, 1)}
        for f in sorted(PUBLIC.iterdir())
    ]
)
display(inventory)
print(f"total {inventory['kilobytes'].sum() / 1024:.2f} MB (Cloudflare's per-file limit is 25 MiB)")
assert inventory["kilobytes"].max() < 25 * 1024

,file,kilobytes
0,calibration.json,0.2
1,cells.json,165.4
2,evidence.json,12.3
3,explanations.json,11.7
4,fixtures.json,372.5
5,holidays.json,2.2
6,model.json,1813.1
7,model_card.json,2.4
8,scenarios.json,3.2
9,story.json,19.4


total 2.35 MB (Cloudflare's per-file limit is 25 MiB)
